# v7: Depot shift: why test looks riskier, and how to correct for it

**Version:** v7  
**Kaggle submission:** **#8** `sub08-2026-11-21-v7_k4` → `outputs/submission_v7_k4.csv` (public score 19,067.17; uploaded by mistake instead of `v9_k4`, but informative: it isolates the shift on the v5 cost model). `v7_em` was not uploaded. v7 is superseded by v8/v9, which add a better cost model to the same correction

**Question:** the model predicts ~3× more failures on the new depot (mean p 0.018 vs 0.006). Is that real, and how should decisions change? Tested: importance weighting, a prior-shift (EM) correction validated on a simulated shift, consistency with the public scores already obtained, the cost of guessing the shift wrongly, and whether a better classifier is the answer instead.

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
def em_odds_factor(p, pi_tr, it=300):
    """Saerens-Latinne-Decaestecker EM: odds factor k by which the test failure odds exceed the training odds."""
    pi = pi_tr
    for _ in range(it):
        a = pi / pi_tr * p
        pi = (a / (a + (1 - pi) / (1 - pi_tr) * (1 - p))).mean()
    return (pi / (1 - pi)) / (pi_tr / (1 - pi_tr))

In [4]:
def pipeline_costs(oofs, base="A", cm=CM, X=None, repeats=None):
    """Paired CV cost for several failure-probability candidates through the v5 pipeline
    (nested cost model fitted once per fold, Platt fitted on the other folds' OOF)."""
    X = dev[S] if X is None else X
    cost = {k: [] for k in oofs}
    for a, b in F:
        fa = a[y[a] == 1]
        m_b, resid = cost_model(cm, X.iloc[fa], np.log(c[fa]), X.iloc[b])
        for k, o in oofs.items():
            fl, bu = decide_pv(make_cal("platt").fit(o[a], y[a]).predict(o[b]), m_b, resid)
            cost[k].append(per_vehicle(c[b], fl, bu))
    return {k: np.array(v) for k, v in cost.items()}

def report(cost, base):
    A = cost[base]
    for k, x in cost.items():
        d = x - A; print(f"{k:12s} CV cost {x.mean():.4f} ± {x.std():.3f}   paired vs {base}: {d.mean():+.4f}   per fold {np.round(d, 3)}   better {int((d < 0).sum())}/{len(d)}")

### 1. The setting

In [5]:
oof = fit_oof(dev[S]); p_te = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure).predict_proba(te[S])[:, 1]
pl = Platt().fit(oof, y); q_dev, q_te = pl.predict(oof), pl.predict(p_te)
print(f"train failure rate {tr.failure.mean():.4f}")
print(f"mean p   raw: dev OOF {oof.mean():.4f}  test {p_te.mean():.4f}     Platt: dev {q_dev.mean():.4f}  test {q_te.mean():.4f}")
for t in (0.025, 0.1, 0.5): print(f"  share with Platt p > {t}: dev {np.mean(q_dev > t):.4f}   test {np.mean(q_te > t):.4f}")

train failure rate 0.0071
mean p   raw: dev OOF 0.0058  test 0.0183     Platt: dev 0.0071  test 0.0203
  share with Platt p > 0.025: dev 0.0304   test 0.0675
  share with Platt p > 0.1: dev 0.0148   test 0.0408
  share with Platt p > 0.5: dev 0.0041   test 0.0163


### 2. Where does the extra test risk come from? (SHAP contribution, test minus dev)

In [6]:
bst = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure).get_booster()
cd = bst.predict(xgb.DMatrix(dev[S]), pred_contribs=True).mean(0); ct = bst.predict(xgb.DMatrix(te[S]), pred_contribs=True).mean(0)
diff = pd.Series(ct - cd, index=S + ["bias"]).drop("bias").sort_values(ascending=False)
DR = ["s120", "s122", "s126", "s128"]
print(f"mean logit test - dev: {diff.sum():+.3f}   of which the 4 drifted sensors: {diff[DR].sum():+.3f}")
print("largest contributors:", diff.head(6).round(3).to_dict())

mean logit test - dev: +0.339   of which the 4 drifted sensors: +0.133
largest contributors: {'s122': 0.1340000033378601, 's123': 0.041999999433755875, 's149': 0.03400000184774399, 's034': 0.02800000086426735, 's111': 0.01600000075995922, 's053': 0.010999999940395355}


### 3. Importance weighting (p(test|x) / p(train|x)): is there enough overlap?

In [7]:
Sa = [s for s in S if s not in DR]
Xa = pd.concat([tr[Sa], te[Sa]]); z = np.r_[np.zeros(len(tr)), np.ones(len(te))]
adv = cross_val_predict(xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, n_jobs=-1, random_state=SEED), Xa, z,
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:len(tr), 1]
for clip in (None, 20, 5):
    w = adv / (1 - adv + 1e-9)
    if clip: w = np.minimum(w / w.mean(), clip)
    wf = w[tr.failure.to_numpy() == 1]
    print(f"clip {clip}: effective sample size {w.sum() ** 2 / (w ** 2).sum():,.0f} of {len(tr):,}   failures {wf.sum() ** 2 / (wf ** 2).sum():.0f} of {len(wf)}")

clip None: effective sample size 172 of 56,000   failures 1 of 397
clip 20: effective sample size 621 of 56,000   failures 2 of 397
clip 5: effective sample size 725 of 56,000   failures 3 of 397


### 4. Prior-shift correction (Saerens-Latinne-Decaestecker EM)
Assumes only the failure rate changed ($p(x|y)$ fixed). EM estimates the test failure odds from the test predictions alone; each calibrated probability is then shifted by the odds factor $k$: $p'=kp/(1-p+kp)$.

In [8]:
pi_tr = y.mean()
print(f"EM odds factor on test (v2 classifier): k = {em_odds_factor(q_te, pi_tr):.2f}   (sanity: on dev OOF itself k = {em_odds_factor(q_dev, pi_tr):.2f}, should be ~1)")

EM odds factor on test (v2 classifier): k = 6.53   (sanity: on dev OOF itself k = 1.00, should be ~1)


**Validation on a simulated shift inside dev** (keep all failures, subsample healthy vehicles at 1/k_true: pure label shift). Compared: no correction, EM, and the oracle that knows k_true.

In [9]:
rng = np.random.default_rng(SEED); parts = []
for a, b in F:
    fa = a[y[a] == 1]; m_b, resid = cost_model(CM, dev.loc[fa, S], np.log(c[fa]), dev.loc[b, S])
    parts.append((b, make_cal("platt").fit(oof[a], y[a]).predict(oof[b]), m_b, resid, y[a].mean()))
print("k_true   EM k (mean [min,max])      cost/veh: none    EM     oracle    EM gain vs none (folds x reps better)")
for kt in (1, 2, 4, 6):
    est, res = [], {"none": [], "EM": [], "oracle": []}
    for rep in range(4):
        for b, q_b, m_b, resid, pit in parts:
            keep = (y[b] == 1) | (rng.random(len(b)) < 1 / kt)
            q, mm, cc = q_b[keep], m_b[keep], c[b][keep]; kh = em_odds_factor(q, pit); est.append(kh)
            for name, kk in (("none", 1), ("EM", kh), ("oracle", kt)):
                fl, bu = decide_pv(shift(q, kk), mm, resid); res[name].append(per_vehicle(cc, fl, bu))
    r = {k: np.array(v) for k, v in res.items()}; g = r["EM"] - r["none"]
    print(f"{kt:>5}   {np.mean(est):5.2f} [{np.min(est):.2f}, {np.max(est):.2f}]            {r['none'].mean():.3f}   {r['EM'].mean():.3f}   {r['oracle'].mean():.3f}     {g.mean():+.3f} ({int((g < 0).sum())}/{len(g)})")

k_true   EM k (mean [min,max])      cost/veh: none    EM     oracle    EM gain vs none (folds x reps better)


    1    0.99 [0.78, 1.38]            0.904   0.912   0.904     +0.009 (8/20)


    2    1.93 [1.42, 2.67]            1.513   1.406   1.393     -0.107 (12/20)


    4    3.89 [3.03, 4.95]            2.715   2.311   2.284     -0.404 (13/20)


    6    5.91 [4.81, 7.62]            3.908   3.069   3.029     -0.839 (16/20)


### 5. Which shift is consistent with the public scores we already have?
Using our own failure probabilities and the v5 cost model, compute each already-submitted file's **expected** score on a random 30% of test under different assumed shifts k, and compare with the observed public scores. This uses the leaderboard only to estimate **one scalar** (the test failure level), not to fit anything per vehicle.

In [10]:
f_all = tr[tr.failure == 1]; m5, resid5 = cost_model(CM, f_all[S], np.log(f_all.repair_cost.to_numpy()), te[S])
M = np.exp(m5[:, None] + np.quantile(resid5, (np.arange(99) + .5) / 99)[None, :]); mu = M.mean(1)
def exp_cost(fl, b, p):
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return np.where(fl == 1, INSPECT + p * fail + (1 - p) * OVER * b, p * mu)
obs = {"v2": 74933, "v3": 27760, "v3b": 29530, "v4_platt": 23876, "v5_raw": 26061, "v5_platt": 22969}
subs = {k: pd.read_csv(f"../submissions/submission_{k}.csv") for k in obs}
beliefs = {"k=1 (Platt)": q_te, **{f"k={k}": shift(q_te, k) for k in (2, 4, 6.5, 10)}}
tab = pd.DataFrame({b: {s: 0.3 * exp_cost(d.flag.to_numpy(), d.budget.to_numpy(), p).sum() for s, d in subs.items()} for b, p in beliefs.items()}).round(0)
tab.insert(0, "observed", pd.Series(obs)); print(tab.to_string())
o = pd.Series(obs); L = np.log(tab.drop(columns="observed").astype(float)).sub(np.log(o), axis=0)
print("\nRMS log error of the levels:      ", L.pow(2).mean().pow(.5).round(3).to_dict())
print("scale-free (demeaned) RMS:        ", (L - L.mean()).pow(2).mean().pow(.5).round(3).to_dict())
print("implied public-set size factor (1 = 30% of test):", np.exp(-L.mean()).round(2).to_dict())

          observed  k=1 (Platt)      k=2      k=4    k=6.5      k=10
v2           74933      28970.0  44879.0  66812.0  86432.0  107290.0
v3           27760      13964.0  19891.0  29969.0  40827.0   54083.0
v3b          29530      14800.0  21661.0  33213.0  45453.0   60124.0
v4_platt     23876      13544.0  18220.0  26120.0  34688.0   45309.0
v5_raw       26061      11989.0  17235.0  26330.0  36212.0   48330.0
v5_platt     22969      11588.0  15491.0  22237.0  29626.0   38819.0

RMS log error of the levels:       {'k=1 (Platt)': 0.735, 'k=2': 0.381, 'k=4': 0.084, 'k=6.5': 0.334, 'k=10': 0.598}
scale-free (demeaned) RMS:         {'k=1 (Platt)': 0.117, 'k=2': 0.079, 'k=4': 0.08, 'k=6.5': 0.096, 'k=10': 0.117}
implied public-set size factor (1 = 30% of test): {'k=1 (Platt)': 2.07, 'k=2': 1.45, 'k=4': 0.98, 'k=6.5': 0.73, 'k=10': 0.56}


### 6. Regret of guessing k wrongly (expected cost / vehicle on test; rows = k used to build the decisions, columns = the true k)

In [11]:
def build_cost(kb, kt):
    fl, bu = decide_pv(shift(q_te, kb), m5, resid5); return exp_cost(fl, bu, shift(q_te, kt)).mean()
K = [1, 2, 3, 4, 5, 6.5, 8, 10]
T = pd.DataFrame({f"true {kt}": {f"build {kb}": build_cost(kb, kt) for kb in K} for kt in K}); R = T - T.min(axis=0)
print(R.round(3).to_string()); print("\nmax regret over true k in [1, 10]:", R.max(axis=1).round(3).to_dict())

           true 1  true 2  true 3  true 4  true 5  true 6.5  true 8  true 10
build 1     0.000   0.110   0.308   0.533   0.768     1.132   1.496    1.973
build 2     0.077   0.000   0.034   0.110   0.207     0.381   0.572    0.837
build 3     0.176   0.029   0.000   0.019   0.063     0.164   0.286    0.468
build 4     0.272   0.082   0.016   0.000   0.012     0.068   0.150    0.281
build 5     0.363   0.143   0.051   0.011   0.000     0.024   0.076    0.170
build 6.5   0.506   0.251   0.126   0.056   0.016     0.000   0.013    0.058
build 8     0.623   0.346   0.200   0.111   0.054     0.011   0.000    0.015
build 10    0.760   0.462   0.297   0.190   0.115     0.049   0.014    0.000

max regret over true k in [1, 10]: {'build 1': 1.973, 'build 2': 0.837, 'build 3': 0.468, 'build 4': 0.281, 'build 5': 0.363, 'build 6.5': 0.506, 'build 8': 0.623, 'build 10': 0.76}


### 7. Is a better classifier the answer instead? (tuned XGBoost, LightGBM, blend; judged on CV cost)

In [12]:
import lightgbm as lgb
TUNED = dict(n_estimators=500, max_depth=3, learning_rate=0.02, subsample=0.8, colsample_bytree=0.5, min_child_weight=1, n_jobs=-1)
LG = dict(n_estimators=400, num_leaves=15, learning_rate=0.03, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, min_child_samples=20, verbose=-1, n_jobs=-1)
cand = {"v2": oof, "tuned x3 seeds": np.zeros(len(dev)), "lightgbm": np.zeros(len(dev))}
for a, b in F:
    for s in (42, 7, 2024):
        cand["tuned x3 seeds"][b] += xgb.XGBClassifier(**TUNED, random_state=s).fit(dev.loc[a, S], y[a]).predict_proba(dev.loc[b, S])[:, 1] / 3
    cand["lightgbm"][b] = lgb.LGBMClassifier(**LG, random_state=SEED).fit(dev.loc[a, S], y[a]).predict_proba(dev.loc[b, S])[:, 1]
cand["xgb + lgbm blend"] = (oof + cand["lightgbm"]) / 2
for k, o in cand.items(): print(f"{k:18s} log-loss {log_loss(y, o):.5f}   AUC {roc_auc_score(y, o):.4f}")
cost = pipeline_costs(cand); print(); report(cost, "v2")

v2                 log-loss 0.01676   AUC 0.9840
tuned x3 seeds     log-loss 0.01589   AUC 0.9854
lightgbm           log-loss 0.02007   AUC 0.9843
xgb + lgbm blend   log-loss 0.01752   AUC 0.9842



v2           CV cost 0.9037 ± 0.235   paired vs v2: +0.0000   per fold [0. 0. 0. 0. 0.]   better 0/5
tuned x3 seeds CV cost 0.9395 ± 0.231   paired vs v2: +0.0358   per fold [0.02  0.032 0.062 0.026 0.038]   better 0/5
lightgbm     CV cost 0.9878 ± 0.306   paired vs v2: +0.0840   per fold [ 0.14   0.05   0.029 -0.015  0.215]   better 1/5
xgb + lgbm blend CV cost 0.9144 ± 0.242   paired vs v2: +0.0107   per fold [ 0.009  0.032  0.002 -0.003  0.014]   better 1/5


### 8. Corrected files (v2 classifier, Platt, v5 cost model, shift k)

In [13]:
for k, name in ((4, "v7_k4"), (em_odds_factor(q_te, pi_tr), "v7_em")):      # EM estimate, unrounded
    fl, bu = decide_pv(shift(q_te, k), m5, resid5); print(f"k={k}: ", end=""); write_submission(te.id, fl, bu, name)

k=4: wrote outputs/submission_v7_k4.csv: flagged 9.45% of test vehicles
k=6.527913553758708: wrote outputs/submission_v7_em.csv: flagged 10.85% of test vehicles


## Results and decisions

- **Importance weighting is ruled out:** effective sample size is **0.3% of rows unclipped, 1.3% clipped; 1-3 of 397 failures** carry any weight. Train and test overlap too little.
- **The extra risk is not one artifact:** s122 contributes +0.13 of the +0.34 mean-logit increase, the rest is diffuse; a model *without* the drifted sensors still predicts the same elevated risk (v3b). This favours a genuinely higher failure rate at the new depot.
- **EM** gives k = **6.53** on test (6.19 with the tuned classifier and 5.67 without the drifted sensors, in `reports/findings.md`): a test failure rate of ~4.5% vs 0.71%. **Validated on simulated shift:** it recovers k (true 2/4/6 → 1.9/3.9/5.9), captures ~95% of the oracle's gain (k=6: none 3.91 → EM 3.07 → oracle 3.03) and costs +0.009 when there is no shift.
- **Public-score consistency:** if the public set is 30% of test, **k ≈ 4** explains the scores almost exactly (level fit 0.084, size factor 0.98); the unshifted model (k=1) fits worst and the scale-free pattern prefers k 2-4. EM's 6.5 sits higher, as expected: under extra covariate shift EM tends to over-estimate.
- **Regret:** under-correcting is expensive, over-correcting cheap (building with k=1 when the truth is 6.5 costs +1.13/vehicle; with 6.5 when the truth is 3 costs +0.13). **k = 4 minimises worst-case regret over k ∈ [1, 10]** (0.28).
- **A better classifier is not the answer:** tuned XGBoost, LightGBM and the blend all improve log-loss/AUC but **none improves CV cost** (tuned: 0.940 vs 0.904, worse in 5/5 folds; LightGBM 0.988; blend 0.914). Cost depends on the few vehicles near the 2.5% threshold, not on overall ranking.

**Caveats**
- EM assumes pure label shift; the test depot also has covariate shift, so the EM k is biased (probably upward).
- The public-score fit uses six public scores to estimate **one scalar**; the public subset is noisy (~6,000 vehicles, few failures).
- A single constant k assumes the extra failures are spread proportionally; if they are concentrated in some vehicle types, flags are misallocated.
- The cost model is assumed unchanged at the new depot (its sensors do not drift, v5).
- There is **no CV number for the shift itself** (dev has no shift by construction); its support is the simulation plus the public-score consistency.

**Next:** v8 and v9 improve the repair-cost model; the shift factor k is then fixed from the observed leaderboard differences between k = 1, 4 and the EM value.